In [15]:
%pip install requests beautifulsoup4 lxml trafilatura pandas openpyxl


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
from __future__ import annotations
 
import argparse
import datetime as dt
import gzip
import json
import logging
import re
import sqlite3
import sys
import threading
import time
import zlib
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, replace
from pathlib import Path
from urllib import robotparser
from urllib.parse import parse_qs, urlencode, urljoin, urlparse, urlunparse, parse_qsl
 
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
 
log = logging.getLogger("caraway")

In [17]:

# ─────────────────────────────────────────────────────────────────────────────
# 1. SETTINGS  (edit these)
# ─────────────────────────────────────────────────────────────────────────────
DEFAULT_EXCEL = "Caraway_Jain_Data_Classification_Sheet.xlsx"
DEFAULT_START = "2024-01-01"
CONTACT_EMAIL = "your.email@mail.utoronto.ca"  # shown to site admins in the User-Agent
 
# Analysis periods, from comment NJ2 in the draft (start inclusive, end exclusive).
PERIODS = [
    ("Pre-withdrawal", "2024-01-01", "2025-01-01"),
    ("Transition", "2025-01-01", "2026-01-01"),
    ("Post-withdrawal", "2026-01-01", "2100-01-01"),
]
 
# Codeword categories that count as a "strong" match on their own. Broad
# indicator words (energy, green, climate...) only count if 2+ distinct appear.
BROAD_CATEGORY_HINT = "adjective"
MIN_BROAD_TERMS_FOR_STRONG = 3
 
SOURCE_TYPE_LABELS = {
    "gov": "Govt press release",
    "ministry": "Ministry of Environment",
    "news": "News",
}
 
# Per-site settings, matched by URL prefix against URLs from the spreadsheet.
# Fields: see the Source dataclass below. "verify" in notes = my best guess at
# the site's structure; run --probe and adjust if it finds 0 links.
SOURCE_OVERRIDES: dict[str, dict] = {
    # --- UK: GOV.UK has an official JSON search API; far more reliable than scraping.
    "https://www.gov.uk/search/": dict(
        strategy="govuk_api",
        notes="GOV.UK Search API. Any gov.uk search URL with organisations[]=... works."),
    # --- Hong Kong: one index page per day, e.g. /gia/general/202607/27.htm
    "https://www.info.gov.hk/gia/general/": dict(
        strategy="hk_daily", title_prefilter=True,
        link_pattern=r"/gia/general/\d{6}/\d{2}/P\d+\.htm",
        notes="Daily index pages; ~100 releases/day, so only keyword-matching titles are fetched."),
    # --- USA
    "https://www.epa.gov/newsreleases": dict(
        strategy="listing", page_param="page", page_start=0,
        link_pattern=r"epa\.gov/newsreleases/[^/?#]+/?$", notes="Drupal ?page=N (verify)"),
    "https://www.state.gov/": dict(
        notes="Pre-Jan-2025 releases were moved to an archive domain; see EXTRA_SOURCES."),
    # --- Canada
    "https://www.pm.gc.ca/en/news": dict(
        strategy="listing", page_param="page", page_start=0,
        link_pattern=r"/en/news/[^/]+/\d{4}/\d{2}/\d{2}/", notes="Drupal ?page=N (verify)"),
    "https://www.canada.ca/en/news/advanced-news-search": dict(
        strategy="listing", page_param="idx", page_start=0, page_step=10,
        fixed_params={"start": "{start}", "end": "{end}"},
        link_pattern=r"/news/\d{4}/\d{2}/", notes="date-filtered search, idx offset (verify)"),
    # --- EU
    "https://environment.ec.europa.eu/news": dict(
        strategy="listing", page_param="page", page_start=0,
        link_pattern=r"environment\.ec\.europa\.eu/news/[^/?#]+_en$", notes="Drupal ?page=N (verify)"),
    "https://ec.europa.eu/commission/presscorner": dict(
        render_js=True, notes="JavaScript app: needs --js (Playwright). Verify."),
    # --- China
    "https://english.mee.gov.cn/": dict(
        strategy="listing", page_url_template="{base}index_{n}.shtml", page_start=1,
        link_pattern=r"/t\d{8}_\d+\.s?html?$", notes="index_N.shtml pagination (verify)"),
    "https://english.www.gov.cn/news": dict(
        link_pattern=r"/news/\d{6}/\d{2}/content_", notes="URL carries the date (verify)"),
    # --- New Zealand
    "https://www.beehive.govt.nz/releases": dict(
        strategy="listing", page_param="page", page_start=0,
        link_pattern=r"beehive\.govt\.nz/release/[^/?#]+$", notes="Drupal ?page=N (verify)"),
    # --- Singapore / Korea: likely JavaScript-rendered
    "https://www.sgpc.gov.sg": dict(render_js=True, notes="JavaScript-rendered (verify)"),
    "https://www.korea.net/Government/Briefing-Room": dict(notes="verify pagination with --probe"),
    "https://mcee.go.kr/": dict(notes="Korean boards often open posts via JS onclick; may need a custom handler"),
    # --- India: PIB listing pages are ASP.NET forms driven by postbacks
    "https://www.pib.gov.in/": dict(
        enabled=False, notes="ASP.NET postback forms; needs a custom handler. Disabled for now."),
}
 
# Sources that are not in the spreadsheet but are needed for full coverage.
EXTRA_SOURCES: list[dict] = [
    dict(country="USA", source_type="Govt press release",
         url="https://2021-2025.state.gov/press-releases/",
         notes="Archive of State Dept pages from before 20 Jan 2025 (verify URL)."),
]
 

In [18]:

# ─────────────────────────────────────────────────────────────────────────────
# 2. DATA CLASSES
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Source:
    country: str
    source_type: str
    url: str
    strategy: str = "auto"            # auto | sitemap | listing | govuk_api | hk_daily
    link_pattern: str | None = None   # regex an article URL must match
    page_param: str | None = None     # e.g. "page" -> ?page=0,1,2...
    page_start: int = 0
    page_step: int = 1
    page_url_template: str | None = None  # e.g. "{base}index_{n}.shtml"
    fixed_params: dict = field(default_factory=dict)  # extra query params ({start}/{end} filled in)
    title_prefilter: bool = False     # only fetch links whose title matches a codeword
    render_js: bool = False           # needs a headless browser (--js)
    enabled: bool = True
    notes: str = ""
 
    @property
    def site(self) -> str:
        return urlparse(self.url).netloc.lower().removeprefix("www.")
 
 
@dataclass
class Candidate:
    url: str
    title_hint: str = ""
    date: dt.date | None = None
    precision: str = ""      # "day" | "month"
    date_source: str = ""    # where the date came from
 

In [19]:

# ─────────────────────────────────────────────────────────────────────────────
# 3. READ THE CLASSIFICATION SHEET
# ─────────────────────────────────────────────────────────────────────────────
URL_RE = re.compile(r"https?://[^\s,;]+")
 
 
def _find_col(columns, *prefixes):
    for c in columns:
        name = str(c).strip().lower()
        if any(name.startswith(p) for p in prefixes):
            return c
    return None
 
def read_sheet(xlsx, sheet):
    """Read one tab from the .xlsx if it exists; otherwise from a CSV with the tab name in its file name."""
    p = Path(xlsx)
    if p.exists() and p.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(p, sheet_name=sheet, dtype=str)
    folder = p if p.is_dir() else p.parent
    matches = [f for f in folder.glob("*.csv") if sheet.lower() in f.name.lower()]
    if not matches:
        found = [f.name for f in folder.glob("*.csv")]
        raise FileNotFoundError(f"No '{p.name}' and no CSV with '{sheet}' in its name in "
                                f"{folder.resolve()}. CSVs there: {found}")
    for enc in ("utf-8-sig", "cp1252"):   # Excel on Windows often saves CSVs as cp1252
        try:
            return pd.read_csv(matches[0], dtype=str, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {matches[0]}")

def load_sources(xlsx: str) -> list[Source]:
    df = read_sheet(xlsx, "General")
    country_col = _find_col(df.columns, "countries")
    gov_col = _find_col(df.columns, "govt press", "government press")
    min_col = _find_col(df.columns, "ministry")
    news_cols = [c for c in df.columns if str(c).strip().lower().startswith("news")]
 
    sources, seen = [], set()
 
    def add(country, stype, cell):
        if not isinstance(cell, str):
            return
        for url in URL_RE.findall(cell):
            url = url.rstrip(".,)")
            if url in seen:
                continue
            seen.add(url)
            sources.append(apply_overrides(Source(country=country, source_type=stype, url=url)))
 
    for _, row in df.iterrows():
        country = row.get(country_col)
        if isinstance(country, str) and country.strip():
            add(country.strip(), SOURCE_TYPE_LABELS["gov"], row.get(gov_col))
            add(country.strip(), SOURCE_TYPE_LABELS["ministry"], row.get(min_col))
        for nc in news_cols:  # news outlets are not tied to a country in the sheet
            add("International (news)", SOURCE_TYPE_LABELS["news"], row.get(nc))
 
    for extra in EXTRA_SOURCES:
        if extra["url"] not in seen:
            sources.append(apply_overrides(Source(**extra)))
    return sources
 
 
def apply_overrides(src: Source) -> Source:
    best = None
    for prefix in SOURCE_OVERRIDES:
        if src.url.startswith(prefix) and (best is None or len(prefix) > len(best)):
            best = prefix
    if best:
        opts = dict(SOURCE_OVERRIDES[best])
        if src.notes and "notes" in opts:
            opts["notes"] = src.notes + " " + opts["notes"]
        src = replace(src, **opts)
    return src
 

In [20]:

# ─────────────────────────────────────────────────────────────────────────────
# 4. KEYWORDS
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Keyword:
    term: str
    category: str
    country: str | None
    pattern: re.Pattern
 
 
def _keyword_pattern(term: str) -> re.Pattern:
    # Acronyms (ESG, COP, GRI...) are case-sensitive so "cop" or "agri" don't match.
    # COP also matches COP28, COP 29 etc.
    if re.fullmatch(r"[A-Z0-9&]{2,8}", term):
        return re.compile(rf"(?<![A-Za-z]){re.escape(term)}\d*(?![A-Za-z])")
    words = [re.escape(w) for w in term.split()]
    body = r"[\s\-]+".join(words)  # "net zero" also matches "net-zero"
    return re.compile(rf"\b{body}(?:s|es)?\b", re.IGNORECASE)
 
 
def load_keywords(xlsx: str) -> list[Keyword]:
    df = read_sheet(xlsx, "Codewords")
    categories = [c for c in df.columns[:4]]
    country_col = df.columns[4] if len(df.columns) > 4 else None
    if country_col is not None:
        df[country_col] = df[country_col].ffill()  # merged cells -> fill down
 
    keywords, seen = [], set()
    for _, row in df.iterrows():
        for cat in categories:
            cell = row.get(cat)
            if not isinstance(cell, str) or not cell.strip():
                continue
            country = row.get(country_col) if (country_col is not None and cat == categories[3]) else None
            # "European Environment Agency (EEA)" -> both the name and the acronym
            m = re.match(r"^(.*?)\s*\(([^)]+)\)\s*$", cell.strip())
            terms = [m.group(1), m.group(2)] if m else [cell.strip()]
            for t in terms:
                key = (t.lower(), cat)
                if t and key not in seen:
                    seen.add(key)
                    keywords.append(Keyword(t, str(cat), country if isinstance(country, str) else None,
                                            _keyword_pattern(t)))
    return keywords
 
 
def match_keywords(text: str, keywords: list[Keyword]) -> dict:
    by_cat: dict[str, set] = defaultdict(set)
    person_countries = set()
    for kw in keywords:
        if kw.pattern.search(text):
            by_cat[kw.category].add(kw.term)
            if kw.country:
                person_countries.add(kw.country)
    distinct = sum(len(v) for v in by_cat.values())
    strong = any(len(v) for c, v in by_cat.items() if BROAD_CATEGORY_HINT not in c.lower())
    for c, v in by_cat.items():
        if BROAD_CATEGORY_HINT in c.lower():
            # "renewable energy" already implies "energy": count only the most specific terms
            specific = {t for t in v if not any(t != o and t.lower() in o.lower() for o in v)}
            if len(specific) >= MIN_BROAD_TERMS_FOR_STRONG:
                strong = True
    return {"by_cat": by_cat, "distinct": distinct, "strong": strong,
            "person_countries": sorted(person_countries)}
 

In [21]:

# ─────────────────────────────────────────────────────────────────────────────
# 5. DATES
# ─────────────────────────────────────────────────────────────────────────────
def _mkdate(y, m, d=1):
    try:
        out = dt.date(int(y), int(m), int(d))
        return out if 2000 <= out.year <= 2100 else None
    except ValueError:
        return None
 
 
def date_from_url(url: str) -> tuple[dt.date | None, str]:
    path = urlparse(url).path
    for rx in (r"(?<!\d)(20\d{2})[/-](\d{2})[/-](\d{2})(?!\d)",   # /2024/05/12/  2024-05-12
               r"(?<!\d)(20\d{2})(\d{2})/(\d{2})(?!\d)",           # /202405/12/ (HK, gov.cn)
               r"(?<![\dA-Za-z])t?(20\d{2})(\d{2})(\d{2})(?!\d)"):  # t20240512_ (Chinese gov)
        m = re.search(rx, path)
        if m and (d := _mkdate(*m.groups())):
            return d, "day"
    m = re.search(r"(?<!\d)(20\d{2})/(\d{2})(?!\d)", path)
    if m and (d := _mkdate(*m.groups())):
        return d, "month"
    for seg in path.split("/"):
        m = re.fullmatch(r"(20\d{2})(\d{2})", seg)
        if m and (d := _mkdate(*m.groups())):
            return d, "month"
    return None, ""
 
 
_MONTHS = ("jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec|january|february|march|april|june|"
           "july|august|september|october|november|december")
_TEXT_DATE_RES = [
    (re.compile(r"(20\d{2})[-./](\d{1,2})[-./](\d{1,2})"), "ymd"),
    (re.compile(rf"(\d{{1,2}})(?:st|nd|rd|th)?\s+({_MONTHS})\.?,?\s+(20\d{{2}})", re.I), "dmy"),
    (re.compile(rf"({_MONTHS})\.?\s+(\d{{1,2}}),?\s+(20\d{{2}})", re.I), "mdy"),
]
 
 
def date_from_text(s: str) -> dt.date | None:
    for rx, kind in _TEXT_DATE_RES:
        m = rx.search(s)
        if not m:
            continue
        try:
            if kind == "ymd":
                return _mkdate(*m.groups())
            if kind == "dmy":
                d, mon, y = m.groups()
            else:
                mon, d, y = m.groups()
            return dt.datetime.strptime(f"{d} {mon[:3]} {y}", "%d %b %Y").date()
        except ValueError:
            continue
    return None
 
 
def month_end(d: dt.date) -> dt.date:
    nxt = dt.date(d.year + (d.month == 12), d.month % 12 + 1, 1)
    return nxt - dt.timedelta(days=1)
 
 
def outside_range(d, precision, start, end) -> bool:
    if d is None:
        return False
    if precision == "month":
        return month_end(d) < start or d.replace(day=1) > end
    return d < start or d > end
 
 
def period_of(d: dt.date | None) -> str:
    if d is None:
        return ""
    for label, a, b in PERIODS:
        if dt.date.fromisoformat(a) <= d < dt.date.fromisoformat(b):
            return label
    return ""
 

In [22]:

# ─────────────────────────────────────────────────────────────────────────────
# 6. POLITE, CACHED FETCHING
# ─────────────────────────────────────────────────────────────────────────────
TRACKING = re.compile(r"^(utm_|fbclid|gclid|mc_)")
 
 
def normalize_url(url: str) -> str:
    p = urlparse(url)
    q = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=True) if not TRACKING.match(k)]
    return urlunparse((p.scheme, p.netloc.lower(), p.path, "", urlencode(q), ""))
 
 
class Fetcher:
    """requests + robots.txt + per-domain delay + SQLite cache (thread-safe)."""
 
    def __init__(self, cache_path: str, delay: float, offline=False, use_js=False, email=CONTACT_EMAIL):
        self.delay, self.offline, self.use_js = delay, offline, use_js
        self.ua = f"CarawayJainResearch/0.1 (academic research; contact: {email})"
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": self.ua, "Accept-Language": "en"})
        self.db = sqlite3.connect(cache_path, check_same_thread=False)
        self.db.execute("CREATE TABLE IF NOT EXISTS pages (url TEXT PRIMARY KEY, fetched REAL, body BLOB)")
        self.db_lock = threading.Lock()
        self.domain_locks: dict[str, threading.Lock] = defaultdict(threading.Lock)
        self.last_hit: dict[str, float] = {}
        self.robots: dict[str, robotparser.RobotFileParser | None] = {}
        self.robots_text: dict[str, str] = {}
        self.stats = Counter()
        self._js = threading.local()
        self._browsers = []
 
    # -- cache
    def _cache_get(self, url, max_age):
        with self.db_lock:
            row = self.db.execute("SELECT fetched, body FROM pages WHERE url=?", (url,)).fetchone()
        if row and (max_age is None or self.offline or time.time() - row[0] < max_age):
            return zlib.decompress(row[1]).decode("utf-8", "replace")
        return None
 
    def _cache_put(self, url, text):
        with self.db_lock:
            self.db.execute("INSERT OR REPLACE INTO pages VALUES (?,?,?)",
                            (url, time.time(), zlib.compress(text.encode("utf-8"))))
            self.db.commit()
 
    # -- robots.txt
    def allowed(self, url) -> bool:
        p = urlparse(url)
        base = f"{p.scheme}://{p.netloc}"
        if base not in self.robots:
            rp = None
            try:
                txt = self._raw_get(base + "/robots.txt", respect_robots=False)
                if txt is not None:
                    rp = robotparser.RobotFileParser()
                    rp.parse(txt.splitlines())
                    self.robots_text[base] = txt
            except Exception:
                rp = None  # no readable robots.txt -> treat as allowed
            self.robots[base] = rp
        rp = self.robots[base]
        return True if rp is None else rp.can_fetch(self.ua, url)
 
    def sitemaps_for(self, url) -> list[str]:
        p = urlparse(url)
        base = f"{p.scheme}://{p.netloc}"
        self.allowed(url)
        txt = self.robots_text.get(base, "")
        return re.findall(r"(?im)^\s*sitemap:\s*(\S+)", txt)
 
    # -- network
    def _throttle(self, netloc):
        with self.domain_locks[netloc]:
            wait = self.last_hit.get(netloc, 0) + self.delay - time.time()
            if wait > 0:
                time.sleep(wait)
            self.last_hit[netloc] = time.time()
 
    def _raw_get(self, url, respect_robots=True):
        if self.offline:
            return None
        if respect_robots and not self.allowed(url):
            self.stats["blocked_by_robots"] += 1
            log.debug("robots.txt disallows %s", url)
            return None
        netloc = urlparse(url).netloc
        for attempt in range(3):
            self._throttle(netloc)
            try:
                r = self.session.get(url, timeout=30)
            except requests.RequestException as e:
                log.debug("error %s: %s", url, e)
                time.sleep(2 ** attempt)
                continue
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(5 * (attempt + 1))
                continue
            if r.status_code != 200:
                self.stats[f"http_{r.status_code}"] += 1
                return None
            body = r.content
            if body[:2] == b"\x1f\x8b":  # gzipped sitemap
                body = gzip.decompress(body)
                return body.decode("utf-8", "replace")
            return r.text
        self.stats["failed"] += 1
        return None
 
    def _render_js(self, url):
        if not hasattr(self._js, "browser"):
            from playwright.sync_api import sync_playwright  # optional dependency
            self._js.pw = sync_playwright().start()
            self._js.browser = self._js.pw.chromium.launch()
            self._browsers.append((self._js.pw, self._js.browser))
        if not self.allowed(url):
            return None
        self._throttle(urlparse(url).netloc)
        page = self._js.browser.new_page(user_agent=self.ua)
        try:
            page.goto(url, wait_until="networkidle", timeout=45000)
            return page.content()
        except Exception as e:
            log.debug("js render failed %s: %s", url, e)
            return None
        finally:
            page.close()
 
    def get(self, url, max_age=None, render_js=False) -> str | None:
        """max_age in seconds; None = cache forever (articles never change much)."""
        url = normalize_url(url)
        cached = self._cache_get(url, max_age)
        if cached is not None:
            self.stats["cache_hits"] += 1
            return cached
        if self.offline:
            return None
        if render_js and not self.use_js:
            self.stats["skipped_needs_js"] += 1
            return None
        text = self._render_js(url) if render_js else self._raw_get(url)
        if text is not None:
            self.stats["fetched"] += 1
            self._cache_put(url, text)
        return text
 
    def close(self):
        for pw, browser in self._browsers:
            try:
                browser.close(); pw.stop()
            except Exception:
                pass
        self.db.close()
 
 
DAY = 24 * 3600  # listing pages are re-fetched after a day; articles are cached forever

In [23]:

# ─────────────────────────────────────────────────────────────────────────────
# 7. DISCOVERY STRATEGIES
# ─────────────────────────────────────────────────────────────────────────────
SKIP_PATH = re.compile(r"/(search|login|signin|contact|about|privacy|terms|accessibility|sitemap|"
                       r"subscribe|rss|feed|tag|tags|category|categories|author|topics?)(/|$)", re.I)
SKIP_EXT = re.compile(r"\.(pdf|jpe?g|png|gif|svg|zip|docx?|xlsx?|pptx?|mp4|mp3|xml|json|css|js)$", re.I)
NEXT_TEXT = re.compile(r"^\s*(next|next page|older|›|»|>|next ›|next »)\s*$", re.I)
 
 
def same_site(url, src: Source) -> bool:
    host = urlparse(url).netloc.lower().removeprefix("www.")
    base = src.site
    root = ".".join(base.split(".")[-3:]) if base.endswith((".gov.uk", ".gc.ca", ".gov.cn", ".gov.hk",
                                                            ".govt.nz", ".gov.sg", ".go.kr", ".gov.in",
                                                            ".europa.eu", ".co.uk")) else ".".join(base.split(".")[-2:])
    return host == base or host.endswith("." + root) or host == root
 
 
def looks_like_article(url, src: Source, listing_url: str) -> bool:
    if src.link_pattern:
        return re.search(src.link_pattern, url) is not None
    p = urlparse(url)
    if normalize_url(url) == normalize_url(listing_url) or SKIP_PATH.search(p.path) or SKIP_EXT.search(p.path):
        return False
    if re.search(r"[?&](page|idx|start|p)=", url) or re.search(r"/(page|index)[-_]?\d+(\.\w+)?$", p.path, re.I):
        return False
    segs = [s for s in p.path.split("/") if s]
    if not segs:
        return False
    last = segs[-1]
    return len(segs) >= 2 and (len(last) >= 12 or bool(re.search(r"\d", last)))
 
 
def extract_links(html, page_url, src: Source) -> list[Candidate]:
    soup = BeautifulSoup(html, "lxml")
    out, seen = [], set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("#", "mailto:", "javascript:", "tel:")):
            continue
        rel = a.get("rel") or []
        if "next" in rel or "prev" in rel or NEXT_TEXT.match(a.get_text(" ", strip=True) or "-"):
            continue  # pagination, not an article
        url = normalize_url(urljoin(page_url, href))
        if url in seen or not same_site(url, src) or not looks_like_article(url, src, page_url):
            continue
        seen.add(url)
        title = " ".join(a.get_text(" ", strip=True).split())
        d, prec = date_from_url(url)
        dsrc = "url" if d else ""
        if d is None:  # look for a date printed next to the link
            node = a
            for _ in range(3):
                node = node.parent
                if node is None:
                    break
                txt = node.get_text(" ", strip=True)
                if len(txt) > 400:
                    break
                if (d := date_from_text(txt)):
                    prec, dsrc = "day", "listing page"
                    break
        out.append(Candidate(url, title, d, prec, dsrc))
    return out
 
 
def find_next_link(html, page_url) -> str | None:
    soup = BeautifulSoup(html, "lxml")
    tag = soup.find("a", rel=lambda v: v and "next" in v) or soup.find("link", rel=lambda v: v and "next" in v)
    if tag is None:
        for a in soup.find_all("a", href=True):
            label = a.get_text(" ", strip=True) or a.get("aria-label", "") or a.get("title", "")
            if NEXT_TEXT.match(label) or re.search(r"\bnext\b", a.get("aria-label", ""), re.I):
                tag = a
                break
    if tag is not None and tag.get("href"):
        return normalize_url(urljoin(page_url, tag["href"]))
    return None
 
 
def _with_params(url, params: dict):
    p = urlparse(url)
    q = dict(parse_qsl(p.query, keep_blank_values=True))
    q.update(params)
    return urlunparse((p.scheme, p.netloc, p.path, "", urlencode(q), ""))
 
 
def page_url(src: Source, i: int, start, end) -> str:
    fixed = {k: v.format(start=start.isoformat(), end=end.isoformat()) for k, v in src.fixed_params.items()}
    base = _with_params(src.url, fixed) if fixed else src.url
    if src.page_url_template:
        if i == 0:
            return base
        folder = base if base.endswith("/") else base.rsplit("/", 1)[0] + "/"
        return src.page_url_template.format(base=folder, n=src.page_start + (i - 1) * src.page_step)
    if src.page_param:
        return _with_params(base, {src.page_param: str(src.page_start + i * src.page_step)})
    return base
 
 
def discover_listing(src, fetcher, start, end, max_pages):
    seen = set()
    url = page_url(src, 0, start, end)
    for i in range(max_pages):
        html = fetcher.get(url, max_age=DAY, render_js=src.render_js)
        if not html:
            break
        cands = [c for c in extract_links(html, url, src) if c.url not in seen]
        if not cands:
            break
        for c in cands:
            seen.add(c.url)
            yield c
        dated = [c for c in cands if c.date]
        if dated and all(outside_range(c.date, c.precision, start, dt.date.max) for c in dated):
            break  # the whole page is older than the start date
        if src.page_param or src.page_url_template:
            url = page_url(src, i + 1, start, end)
        else:
            url = find_next_link(html, url)
            if not url:
                break
 
 
def _parse_sitemap(xml):
    soup = BeautifulSoup(xml, "xml")
    children = [(s.loc.text.strip(), s.lastmod.text.strip() if s.lastmod else "")
                for s in soup.find_all("sitemap") if s.loc]
    urls = [(u.loc.text.strip(), u.lastmod.text.strip() if u.lastmod else "")
            for u in soup.find_all("url") if u.loc]
    return children, urls
 
 
def discover_sitemap(src, fetcher, start, end, max_sitemaps=300):
    p = urlparse(src.url)
    base = f"{p.scheme}://{p.netloc}"
    queue = fetcher.sitemaps_for(src.url) or [base + "/sitemap.xml", base + "/sitemap_index.xml",
                                              base + "/wp-sitemap.xml"]
    seen_maps, found = set(), 0
    while queue and len(seen_maps) < max_sitemaps:
        sm = queue.pop(0)
        if sm in seen_maps:
            continue
        seen_maps.add(sm)
        xml = fetcher.get(sm, max_age=DAY)
        if not xml or "<" not in xml[:200]:
            continue
        children, urls = _parse_sitemap(xml)
        for loc, lastmod in children:
            lm = _mkdate(*lastmod[:10].split("-")) if re.match(r"\d{4}-\d{2}-\d{2}", lastmod) else None
            if lm is None or lm >= start:  # a child sitemap untouched since before start can be skipped
                queue.append(loc)
        for loc, lastmod in urls:
            if not same_site(loc, src):
                continue
            if src.link_pattern and not re.search(src.link_pattern, loc):
                continue
            lm = _mkdate(*lastmod[:10].split("-")) if re.match(r"\d{4}-\d{2}-\d{2}", lastmod) else None
            if lm and lm < start:  # last modified before start => published before start
                continue
            d, prec = date_from_url(loc)
            found += 1
            yield Candidate(normalize_url(loc), "", d, prec, "url" if d else "")
    log.debug("%s: sitemap yielded %d urls", src.url, found)
 
 
def discover_auto(src, fetcher, start, end, max_pages):
    """Try the sitemap first (needs a link_pattern to stay on-topic), else crawl listing pages."""
    got_any = False
    if src.link_pattern:
        for c in discover_sitemap(src, fetcher, start, end):
            got_any = True
            yield c
    if not got_any:
        yield from discover_listing(src, fetcher, start, end, max_pages)
 
 
def discover_govuk(src, fetcher, start, end, max_pages):
    qs = parse_qs(urlparse(src.url).query)
    orgs = qs.get("organisations[]") or qs.get("organisations") or []
    api = "https://www.gov.uk/api/search.json"
    for offset in range(0, max_pages * 100, 100):
        params = {"filter_content_purpose_supergroup": "news_and_communications",
                  "filter_public_timestamp": f"from:{start},to:{end}",
                  "order": "-public_timestamp", "count": "100", "start": str(offset),
                  "fields": "title,link,public_timestamp"}
        url = api + "?" + urlencode(params) + "".join(f"&filter_organisations={o}" for o in orgs)
        txt = fetcher.get(url, max_age=DAY)
        if not txt:
            break
        results = json.loads(txt).get("results", [])
        for r in results:
            d = _mkdate(*r["public_timestamp"][:10].split("-")) if r.get("public_timestamp") else None
            yield Candidate(normalize_url(urljoin("https://www.gov.uk", r["link"])),
                            r.get("title", ""), d, "day", "GOV.UK API")
        if len(results) < 100:
            break
 
 
def discover_hk_daily(src, fetcher, start, end, max_pages):
    p = urlparse(src.url)
    base = f"{p.scheme}://{p.netloc}/gia/general/"
    today = dt.date.today()
    d = start
    while d <= end:
        idx = f"{base}{d:%Y%m}/{d:%d}.htm"
        html = fetcher.get(idx, max_age=None if (today - d).days > 2 else DAY)
        if html:
            for c in extract_links(html, idx, src):
                c.date, c.precision, c.date_source = d, "day", "daily index"
                yield c
        d += dt.timedelta(days=1)
 
 
STRATEGIES = {"auto": discover_auto, "listing": discover_listing, "govuk_api": discover_govuk,
              "hk_daily": discover_hk_daily,
              "sitemap": lambda s, f, a, b, m: discover_sitemap(s, f, a, b)}
 

In [24]:

# ─────────────────────────────────────────────────────────────────────────────
# 8. PROCESS ONE SOURCE
# ─────────────────────────────────────────────────────────────────────────────
def resolve_date(cand: Candidate, meta: dict):
    url_d, url_prec = date_from_url(cand.url)
    if url_d and url_prec == "day":
        return url_d, "url"
    if cand.date and cand.date_source in ("GOV.UK API", "daily index"):
        return cand.date, cand.date_source
    md = meta.get("date")
    if md and re.match(r"\d{4}-\d{2}-\d{2}", md):
        if (d := _mkdate(*md[:10].split("-"))):
            return d, "page metadata"
    if cand.date and cand.precision == "day":
        return cand.date, cand.date_source
    if url_d:
        return url_d, "url (month only)"
    return None, ""
 
 
def run_source(src: Source, fetcher: Fetcher, keywords, start, end, max_pages, limit):
    stats = Counter()
    records = []
    discover = STRATEGIES.get(src.strategy, discover_auto)
    seen = set()
    try:
        for cand in discover(src, fetcher, start, end, max_pages):
            if cand.url in seen:
                continue
            seen.add(cand.url)
            stats["candidates"] += 1
            if outside_range(cand.date, cand.precision, start, end):
                stats["skipped_out_of_range"] += 1
                continue
            if src.title_prefilter and cand.title_hint and not match_keywords(cand.title_hint, keywords)["distinct"]:
                stats["skipped_title_prefilter"] += 1
                continue
            if limit and stats["fetched_articles"] >= limit:
                break
            html = fetcher.get(cand.url, render_js=src.render_js)
            if not html:
                stats["fetch_failed"] += 1
                continue
            stats["fetched_articles"] += 1
            raw = trafilatura.extract(html, url=cand.url, output_format="json", with_metadata=True,
                                      include_comments=False, include_tables=True)
            meta = json.loads(raw) if raw else {}
            title = (meta.get("title") or cand.title_hint or "").strip()
            text = (meta.get("text") or "").strip()
            date, date_src = resolve_date(cand, meta)
            if date and not (start <= date <= end):
                stats["dropped_out_of_range"] += 1
                continue
            m = match_keywords(title + "\n" + text, keywords)
            if not m["distinct"]:
                stats["dropped_no_keywords"] += 1
                continue
            stats["kept"] += 1
            records.append({
                "month": f"{date:%Y-%m}" if date else "Undated",
                "date": date, "period": period_of(date), "country": src.country,
                "source_type": src.source_type, "site": src.site, "title": title, "url": cand.url,
                "matched_keywords": "; ".join(sorted(set().union(*m["by_cat"].values()))),
                "by_cat": {k: sorted(v) for k, v in m["by_cat"].items()},
                "keyword_count": m["distinct"], "strong_match": "Yes" if m["strong"] else "No",
                "person_countries": ", ".join(m["person_countries"]),
                "date_source": date_src, "word_count": len(text.split()), "text": text,
            })
    except Exception as e:  # one broken site must not kill the whole run
        log.exception("source failed: %s", src.url)
        stats["error"] = 1
        stats["error_msg"] = str(e)[:200]
    log.info("%-55s candidates=%-5d fetched=%-5d kept=%d", src.url[:55], stats["candidates"],
             stats["fetched_articles"], stats["kept"])
    return records, stats
 


In [25]:

# ─────────────────────────────────────────────────────────────────────────────
# 9. PROBE MODE: quick health check of every source
# ─────────────────────────────────────────────────────────────────────────────
def probe(sources, fetcher, start, end, out_dir: Path):
    rows = []
    for src in sources:
        row = {"country": src.country, "type": src.source_type, "url": src.url, "strategy": src.strategy,
               "enabled": src.enabled, "notes": src.notes}
        if src.strategy in ("govuk_api", "hk_daily"):
            gen = STRATEGIES[src.strategy](src, fetcher, start, end, 1)
            sample = [c for _, c in zip(range(5), gen)]
            row.update(status="ok" if sample else "no results", links_found=len(sample),
                       sample_links=" | ".join(c.url for c in sample[:3]))
        else:
            first = page_url(src, 0, start, end)
            html = fetcher.get(first, max_age=DAY, render_js=src.render_js)
            if not html:
                row.update(status="FETCH FAILED (blocked, JS-only, or robots.txt)", links_found=0)
            else:
                links = extract_links(html, first, src)
                soup = BeautifulSoup(html, "lxml")
                visible = len(soup.get_text(" ", strip=True))
                row.update(status="ok" if links else "0 article links: check link_pattern / JS",
                           links_found=len(links), dated_links=sum(bool(c.date) for c in links),
                           has_next_link=bool(find_next_link(html, first)),
                           looks_js_rendered=visible < 1500 and len(soup.find_all("script")) > 5,
                           sitemaps_in_robots=len(fetcher.sitemaps_for(src.url)),
                           sample_links=" | ".join(c.url for c in links[:3]))
        rows.append(row)
        log.info("%-8s %-60s %s (%s links)", src.country[:8], src.url[:60], row["status"], row.get("links_found"))
    df = pd.DataFrame(rows)
    path = out_dir / "probe_report.csv"
    df.to_csv(path, index=False)
    log.info("Probe report written to %s", path)
    return df
 

In [26]:

# ─────────────────────────────────────────────────────────────────────────────
# 10. OUTPUTS
# ─────────────────────────────────────────────────────────────────────────────
HEADER_FONT = Font(name="Arial", bold=True, color="FFFFFF")
HEADER_FILL = PatternFill("solid", start_color="2F5597")
BODY_FONT = Font(name="Arial", size=10)
LINK_FONT = Font(name="Arial", size=10, color="0563C1", underline="single")
 
ARTICLE_COLS = [  # (header, record key, width)
    ("Month", "month", 10), ("Date", "date", 12), ("Period", "period", 15), ("Country", "country", 16),
    ("Source Type", "source_type", 22), ("Site", "site", 26), ("Title", "title", 60), ("URL", "url", 50),
    ("Matched Keywords", "matched_keywords", 45), ("Keyword Count", "keyword_count", 10),
    ("Strong Match", "strong_match", 10), ("Minister/Person Country", "person_countries", 18),
    ("Date Source", "date_source", 16), ("Word Count", "word_count", 10),
]
 
 
def _clean(v):
    return ILLEGAL_CHARACTERS_RE.sub("", v) if isinstance(v, str) else v
 
 
def _header(ws, headers, widths):
    ws.append(headers)
    for i, (cell, w) in enumerate(zip(ws[1], widths), start=1):
        cell.font, cell.fill = HEADER_FONT, HEADER_FILL
        cell.alignment = Alignment(wrap_text=True, vertical="center")
        ws.column_dimensions[get_column_letter(i)].width = w
    ws.freeze_panes = "A2"
 
 
def _article_sheet(wb, name, records, cat_names):
    ws = wb.create_sheet(name)
    headers = [h for h, _, _ in ARTICLE_COLS] + [f"{c} hits" for c in cat_names]
    widths = [w for _, _, w in ARTICLE_COLS] + [28] * len(cat_names)
    _header(ws, headers, widths)
    url_col = [k for _, k, _ in ARTICLE_COLS].index("url") + 1
    for r in records:
        row = [_clean(r[k]) for _, k, _ in ARTICLE_COLS] + ["; ".join(r["by_cat"].get(c, [])) for c in cat_names]
        ws.append(row)
        n = ws.max_row
        for cell in ws[n]:
            cell.font = BODY_FONT
        link = ws.cell(n, url_col)
        link.hyperlink, link.font = r["url"], LINK_FONT
        ws.cell(n, 2).number_format = "yyyy-mm-dd"
    if records:
        ws.auto_filter.ref = ws.dimensions
    return ws
 
 
def months_between(start, end):
    y, m = start.year, start.month
    while (y, m) <= (end.year, end.month):
        yield f"{y}-{m:02d}"
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)
 
 
def write_outputs(records, source_rows, keywords, sources, args, start, end, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    records.sort(key=lambda r: (r["date"] or dt.date.max, r["country"], r["title"]))
    cat_names = list(dict.fromkeys(k.category for k in keywords))
 
    # corpus with full text, for the preprocessing / NLP steps
    with open(out_dir / "corpus.jsonl", "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps({**{k: v for k, v in r.items() if k != "by_cat"},
                                "date": r["date"].isoformat() if r["date"] else None,
                                "keywords_by_category": r["by_cat"]}, ensure_ascii=False) + "\n")
    pd.DataFrame([{h: r[k] for h, k, _ in ARTICLE_COLS} for r in records]).to_csv(
        out_dir / "articles.csv", index=False)
 
    wb = Workbook()
    wb.remove(wb.active)
 
    # Summary: month x country counts (live COUNTIFS on the All Articles sheet)
    countries = list(dict.fromkeys(s.country for s in sources))
    ws = wb.create_sheet("Monthly Summary")
    _header(ws, ["Month", "Period"] + countries + ["Total", "Strong matches"],
            [10, 15] + [14] * len(countries) + [10, 14])
    last_c = get_column_letter(2 + len(countries))
    for i, month in enumerate(months_between(start, end), start=2):
        per = period_of(dt.date.fromisoformat(month + "-01"))
        row = [month, per]
        for j in range(len(countries)):
            col = get_column_letter(3 + j)
            row.append(f"=COUNTIFS('All Articles'!$A:$A,$A{i},'All Articles'!$D:$D,{col}$1)")
        row.append(f"=SUM(C{i}:{last_c}{i})")
        row.append(f"=COUNTIFS('All Articles'!$A:$A,$A{i},'All Articles'!$K:$K,\"Yes\")")
        ws.append(row)
        for cell in ws[i]:
            cell.font = BODY_FONT
    n = ws.max_row
    ws.append(["Total", ""] + [f"=SUM({get_column_letter(c)}2:{get_column_letter(c)}{n})"
                               for c in range(3, 5 + len(countries))])
    for cell in ws[n + 1]:
        cell.font = Font(name="Arial", size=10, bold=True)
    ws.append([])
    ws.append(["Undated articles", "", f"=COUNTIFS('All Articles'!$A:$A,\"Undated\")"])
    for cell in ws[ws.max_row]:
        cell.font = BODY_FONT
    ws.append([])
    ws.append(["Period", ""] + countries + ["Total"])
    hdr = ws.max_row
    for cell in ws[hdr]:
        cell.font = Font(name="Arial", size=10, bold=True)
    for label, _, _ in PERIODS:
        r = ws.max_row + 1
        ws.append([label, ""] + [f"=COUNTIFS('All Articles'!$C:$C,$A{r},'All Articles'!$D:$D,"
                                 f"{get_column_letter(3 + j)}${hdr})" for j in range(len(countries))]
                  + [f"=SUM(C{r}:{last_c}{r})"])
        for cell in ws[r]:
            cell.font = BODY_FONT
 
    _article_sheet(wb, "All Articles", records, cat_names)
    by_month = defaultdict(list)
    for r in records:
        by_month[r["month"]].append(r)
    for month in sorted(by_month, key=lambda m: (m == "Undated", m)):
        _article_sheet(wb, month, by_month[month], cat_names)
 
    # Source log: which sites worked
    ws = wb.create_sheet("Source Log")
    cols = ["country", "source_type", "url", "strategy", "enabled", "candidates", "fetched_articles",
            "kept", "skipped_out_of_range", "skipped_title_prefilter", "dropped_out_of_range",
            "dropped_no_keywords", "fetch_failed", "error_msg", "notes"]
    _header(ws, [c.replace("_", " ").title() for c in cols], [14, 20, 55, 11, 9] + [12] * 8 + [40, 60])
    for row in source_rows:
        ws.append([_clean(row.get(c, "")) for c in cols])
        for cell in ws[ws.max_row]:
            cell.font = BODY_FONT
 
    ws = wb.create_sheet("Notes")
    notes = [
        ("Run date", dt.date.today().isoformat()),
        ("Date range", f"{start} to {end}"),
        ("Keyword file", args.excel),
        ("Periods (from comment NJ2)", "; ".join(f"{l}: {a} to {b}" for l, a, b in PERIODS)),
        ("Keyword match", "Title + main text, word boundaries; acronyms case-sensitive; COP also matches COP28 etc."),
        ("Strong Match", f"Any hit outside '{BROAD_CATEGORY_HINT}' indicators, or {MIN_BROAD_TERMS_FOR_STRONG}+ "
                         "distinct indicator words (overlaps like energy/renewable energy count once). Broad words alone (energy, green) bring noise; filter on this."),
        ("Date Source", "url > GOV.UK API / HK daily index > page metadata > date printed on listing page."),
        ("Minister/Person Country", "Country of any minister/person codeword found (from Codewords col E)."),
        ("Full text", "outputs/corpus.jsonl (one JSON object per article, with metadata)."),
    ]
    _header(ws, ["Item", "Detail"], [28, 110])
    for k, v in notes:
        ws.append([k, v])
        for cell in ws[ws.max_row]:
            cell.font = BODY_FONT
            cell.alignment = Alignment(wrap_text=True, vertical="top")
 
    path = out_dir / "Caraway_Jain_Articles.xlsx"
    wb.save(path)
    return path
 

In [27]:

# ─────────────────────────────────────────────────────────────────────────────
# 11. MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--excel", default=DEFAULT_EXCEL)
    ap.add_argument("--start", default=DEFAULT_START)
    ap.add_argument("--end", default=dt.date.today().isoformat(), help="default: today")
    ap.add_argument("--out", default="outputs")
    ap.add_argument("--cache", default="caraway_cache.sqlite")
    ap.add_argument("--types", default="gov,ministry", help="any of gov,ministry,news")
    ap.add_argument("--countries", default="", help="comma list, e.g. USA,Canada")
    ap.add_argument("--max-pages", type=int, default=150, help="max listing pages per source")
    ap.add_argument("--limit-per-source", type=int, default=0, help="for trial runs")
    ap.add_argument("--delay", type=float, default=1.5, help="seconds between hits on one site")
    ap.add_argument("--workers", type=int, default=6, help="sources processed in parallel")
    ap.add_argument("--email", default=CONTACT_EMAIL)
    ap.add_argument("--js", action="store_true", help="render JavaScript sites (needs playwright)")
    ap.add_argument("--probe", action="store_true", help="only check each source and report")
    ap.add_argument("--offline", action="store_true", help="use cached pages only")
    ap.add_argument("-v", "--verbose", action="store_true")
    args, _ = ap.parse_known_args(argv)  # parse_known_args: safe inside Jupyter
 
    logging.basicConfig(level=logging.DEBUG if args.verbose else logging.INFO,
                        format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")
    for noisy in ("trafilatura", "htmldate", "urllib3", "charset_normalizer"):
        logging.getLogger(noisy).setLevel(logging.WARNING)
 
    start, end = dt.date.fromisoformat(args.start), dt.date.fromisoformat(args.end)
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)
 
    sources = load_sources(args.excel)
    keywords = load_keywords(args.excel)
    wanted_types = {SOURCE_TYPE_LABELS[t.strip()] for t in args.types.split(",") if t.strip()}
    wanted_countries = {c.strip().lower() for c in args.countries.split(",") if c.strip()}
    sources = [s for s in sources if s.source_type in wanted_types
               and (not wanted_countries or s.country.lower() in wanted_countries)]
    log.info("%d sources, %d keywords, %s to %s", len(sources), len(keywords), start, end)
 
    fetcher = Fetcher(args.cache, args.delay, offline=args.offline, use_js=args.js, email=args.email)
    try:
        if args.probe:
            probe(sources, fetcher, start, end, out_dir)
            return
 
        all_records, source_rows = [], []
        active = [s for s in sources if s.enabled]
        for s in sources:
            if not s.enabled:
                source_rows.append({**vars(s), "error_msg": "disabled"})
        with ThreadPoolExecutor(max_workers=max(1, args.workers)) as pool:
            futures = {pool.submit(run_source, s, fetcher, keywords, start, end,
                                   args.max_pages, args.limit_per_source): s for s in active}
            for fut in as_completed(futures):
                s = futures[fut]
                recs, stats = fut.result()
                all_records.extend(recs)
                source_rows.append({**vars(s), **stats})
 
        # the same article can be listed on two sites; keep the first
        dedup, seen = [], set()
        for r in all_records:
            if r["url"] not in seen:
                seen.add(r["url"]); dedup.append(r)
        source_rows.sort(key=lambda r: (r["country"], r["source_type"]))
        path = write_outputs(dedup, source_rows, keywords, sources, args, start, end, out_dir)
        log.info("Done: %d articles -> %s  | fetch stats: %s", len(dedup), path, dict(fetcher.stats))
    finally:
        fetcher.close()
 
 
if __name__ == "__main__":
    main(["--excel", ".", "--end", "2026-05-31", "--limit-per-source", "20"])
 

21:42:55 INFO 24 sources, 98 keywords, 2024-01-01 to 2026-05-31
21:42:57 INFO https://www.epa.gov/newsreleases/search                 candidates=0     fetched=0     kept=0
21:42:57 INFO https://www.state.gov/press-releases/                   candidates=0     fetched=0     kept=0
21:42:57 INFO https://ec.europa.eu/commission/presscorner/home/en     candidates=0     fetched=0     kept=0
21:42:58 INFO https://www.pm.gc.ca/en/news/releases                   candidates=1     fetched=0     kept=0
21:42:58 INFO https://www.sgpc.gov.sg/                                candidates=0     fetched=0     kept=0
21:42:59 INFO https://english.mee.gov.cn/News_service/news_release/   candidates=32    fetched=0     kept=0
21:43:01 INFO https://moef.gov.in/press_release/update                candidates=0     fetched=0     kept=0
21:43:03 INFO https://environment.ec.europa.eu/news_en                candidates=24    fetched=0     kept=0
21:43:07 INFO https://www.cmab.gov.hk/en/press/press.htm              ca